<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Fraud-Detection-with-Federated-Learning/blob/main/Credit_Card_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, random, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ========================= SEED =========================
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ========================= KAGGLE DOWNLOAD =========================
os.environ['KAGGLE_USERNAME'] = "devindithathsara"
os.environ['KAGGLE_KEY'] = "29219555"
!pip install -q kaggle
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip -d dataset/

df = pd.read_csv('dataset/creditcard.csv')
print("\n=== CREDIT CARD DATASET ===")
print(f"Total: {len(df):,}")
print(f"Fraud: {df['Class'].sum():,}")
print(f"Non-Fraud: {len(df)-df['Class'].sum():,}")

# ========================= PREPROCESS =========================
X = df.drop("Class", axis=1).values.astype("float32")
y = df["Class"].values.astype("float32")

# ========================= IID TRAIN-TEST =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"\nTest set: {len(X_test):,} | Fraud: {int(y_test.sum()):,} | Non-Fraud: {len(y_test)-int(y_test.sum()):,}")

# ========================= DIRICHLET SPLIT =========================
NUM_CLIENTS = 12
alpha = 0.5

def dirichlet_split(X, y, num_clients, alpha):
    clients = [[] for _ in range(num_clients)]
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = (proportions / proportions.sum() * len(idx)).astype(int)
        proportions[-1] = len(idx) - proportions[:-1].sum()
        start = 0
        for i in range(num_clients):
            end = start + proportions[i]
            clients[i].extend(idx[start:end])
            start = end
    return {i: {"X": X[np.array(clients[i])], "y": y[np.array(clients[i])]} for i in range(num_clients)}

clients_train = dirichlet_split(X_train, y_train, NUM_CLIENTS, alpha)
clients_test = {i: {"X": X_test, "y": y_test} for i in range(NUM_CLIENTS)}

print("\nNON-IID CLIENT DISTRIBUTION")
for i in range(NUM_CLIENTS):
    yc = clients_train[i]["y"]
    print(f"Client {i+1:2d}: Total={len(yc):,} | Fraud={int(yc.sum()):,} | Non-Fraud={len(yc)-int(yc.sum()):,}")

# ========================= MODEL =========================
def create_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(X.shape[1],)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

LOCAL_EPOCHS = 5
BATCH_SIZE = 512
NUM_ROUNDS = 20

# ========================= NO FEDERATED LEARNING =========================
print("\n=== NO FEDERATED LEARNING ===")
client_models_noFL = []
noFL_metrics = []

for i in range(NUM_CLIENTS):
    print(f"Training Client {i+1}")
    model = create_model()
    model.fit(clients_train[i]["X"], clients_train[i]["y"],
              epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    client_models_noFL.append(model)

    # Evaluate on global IID test set
    pred = (model.predict(clients_test[i]["X"], verbose=0) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    noFL_metrics.append([acc, prec, rec, f1])
    print(f"  Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

# Save per-client CSV
df_noFL = pd.DataFrame(noFL_metrics, columns=["Accuracy", "Precision", "Recall", "F1"])
df_noFL.to_csv("credit_noFL_metrics.csv", index=False)

# Final average for NoFL
avg_noFL = np.mean(noFL_metrics, axis=0)
print("\n=== NOFL FINAL AVERAGE (across 12 clients) ===")
print(f"Avg Accuracy : {avg_noFL[0]:.4f}")
print(f"Avg Precision: {avg_noFL[1]:.4f}")
print(f"Avg Recall   : {avg_noFL[2]:.4f}")
print(f"Avg F1-score : {avg_noFL[3]:.4f}")

# ========================= FEDAVG =========================
print("\n=== FEDAVG ===")
global_model_fedavg = create_model()
round_metrics_fedavg = []

def fedavg_aggregate(weights_list):
    return [np.mean(w, axis=0) for w in zip(*weights_list)]

for r in range(NUM_ROUNDS):
    client_weights = []
    for i in range(NUM_CLIENTS):
        local_model = create_model()
        local_model.set_weights(global_model_fedavg.get_weights())
        local_model.fit(clients_train[i]["X"], clients_train[i]["y"],
                        epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
        client_weights.append(local_model.get_weights())

    global_model_fedavg.set_weights(fedavg_aggregate(client_weights))

    # Evaluate on global IID test set
    pred = (global_model_fedavg.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    loss_val = global_model_fedavg.evaluate(X_test, y_test, verbose=0)[0]

    round_metrics_fedavg.append([acc, prec, rec, f1])
    print(f"Round {r+1:2d} → Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Loss: {loss_val:.4f}")

# Final average for FedAvg
avg_fedavg = np.mean(round_metrics_fedavg, axis=0)
print("\n=== FEDAVG FINAL AVERAGE (across 20 rounds) ===")
print(f"Avg Accuracy : {avg_fedavg[0]:.4f}")
print(f"Avg Precision: {avg_fedavg[1]:.4f}")
print(f"Avg Recall   : {avg_fedavg[2]:.4f}")
print(f"Avg F1-score : {avg_fedavg[3]:.4f}")

pd.DataFrame(round_metrics_fedavg, columns=["Accuracy","Precision","Recall","F1"]).to_csv("credit_fedavg_metrics.csv", index=False)

# ========================= FEDBN =========================
print("\n=== FEDBN ===")
global_model_fedbn = create_model()
client_models_fedbn = [create_model() for _ in range(NUM_CLIENTS)]
for m in client_models_fedbn:
    m.set_weights(global_model_fedbn.get_weights())

def sync_non_bn(global_model, local_model):
    for g_layer, l_layer in zip(global_model.layers, local_model.layers):
        if "batch_normalization" not in g_layer.name.lower():
            l_layer.set_weights(g_layer.get_weights())

def fedbn_aggregate(client_models, global_model):
    new_weights = []
    for i, layer in enumerate(global_model.layers):
        if "batch_normalization" not in layer.name.lower():
            weights = [m.layers[i].get_weights() for m in client_models]
            avg = [np.mean(w, axis=0) for w in zip(*weights)]
            new_weights.extend(avg)
        else:
            new_weights.extend(layer.get_weights())
    return new_weights

round_metrics_fedbn = []

for r in range(NUM_ROUNDS):
    for i in range(NUM_CLIENTS):
        sync_non_bn(global_model_fedbn, client_models_fedbn[i])
        client_models_fedbn[i].fit(clients_train[i]["X"], clients_train[i]["y"],
                                   epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)

    global_model_fedbn.set_weights(fedbn_aggregate(client_models_fedbn, global_model_fedbn))

    # Evaluate average of 12 client models (FedBN keeps local BN)
    accs, precs, recs, f1s = [], [], [], []
    for m in client_models_fedbn:
        pred = (m.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
        accs.append(accuracy_score(y_test, pred))
        precs.append(precision_score(y_test, pred, zero_division=0))
        recs.append(recall_score(y_test, pred, zero_division=0))
        f1s.append(f1_score(y_test, pred, zero_division=0))

    avg_acc = np.mean(accs)
    avg_prec = np.mean(precs)
    avg_rec = np.mean(recs)
    avg_f1 = np.mean(f1s)

    round_metrics_fedbn.append([avg_acc, avg_prec, avg_rec, avg_f1])
    print(f"Round {r+1:2d} → Avg Acc: {avg_acc:.4f} | Prec: {avg_prec:.4f} | Rec: {avg_rec:.4f} | F1: {avg_f1:.4f}")

# Final average for FedBN
avg_fedbn = np.mean(round_metrics_fedbn, axis=0)
print("\n=== FEDBN FINAL AVERAGE (across 20 rounds) ===")
print(f"Avg Accuracy : {avg_fedbn[0]:.4f}")
print(f"Avg Precision: {avg_fedbn[1]:.4f}")
print(f"Avg Recall   : {avg_fedbn[2]:.4f}")
print(f"Avg F1-score : {avg_fedbn[3]:.4f}")

pd.DataFrame(round_metrics_fedbn, columns=["Accuracy","Precision","Recall","F1"]).to_csv("credit_fedbn_metrics.csv", index=False)

print("\n CREDIT CARD DONE! All CSVs saved with per-round and final averages.")

Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 147MB/s]

Archive:  creditcardfraud.zip
  inflating: dataset/creditcard.csv  

=== CREDIT CARD DATASET ===
Total: 284,807
Fraud: 492
Non-Fraud: 284,315

Test set: 56,962 | Fraud: 98 | Non-Fraud: 56,864

NON-IID CLIENT DISTRIBUTION
Client  1: Total=10,642 | Fraud=67 | Non-Fraud=10,575
Client  2: Total=3,130 | Fraud=7 | Non-Fraud=3,123
Client  3: Total=3,083 | Fraud=6 | Non-Fraud=3,077
Client  4: Total=46,687 | Fraud=71 | Non-Fraud=46,616
Client  5: Total=5,905 | Fraud=5 | Non-Fraud=5,900
Client  6: Total=51,292 | Fraud=0 | Non-Fraud=51,292
Client  7: Total=18,307 | Fraud=29 | Non-Fraud=18,278
Client  8: Total=56,474 | Fraud=59 | Non-Fraud=56,415
Client  9: Total=269 | Fraud=48 | Non-Fraud=221
Client 10: Total=25,994 | Fraud=2 | Non-Fraud=25,992
Client 11: Total=2,015 | Fraud=93 | Non-Fraud=1,922
Client 12: Total=4,047 | Fraud=7 | Non-Fraud=4,040

=== NO FEDERATED LEAR